# ⚡ SmartReco End-to-End Diagnostic Notebook
Run these cells sequentially to verify every layer of your recommendation system pipeline manually.

## 1. Environment & API Key Verification

In [8]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("MESH_API_KEY", "")
base_url = os.getenv("MESHAPI_BASE_URL", "")
chat_model = os.getenv("MESHAPI_CHAT_MODEL", "")
embed_model = os.getenv("MESH_EMBEDDING_MODEL", "")

print(f"✅ MESH_API_KEY: {api_key[:8]}...{api_key[-4:] if len(api_key)>12 else ''}")
print(f"✅ MESHAPI_BASE_URL: {base_url}")
print(f"✅ CHAT MODEL: {chat_model}")
print(f"✅ EMBEDDING MODEL: {embed_model}")

✅ MESH_API_KEY: rsk_01KZ...MWS0
✅ MESHAPI_BASE_URL: https://api.meshapi.ai
✅ CHAT MODEL: openai/gpt-4o-mini
✅ EMBEDDING MODEL: openai/text-embedding-3-small


## 2. Test MeshAPI Embeddings (Vector Search Layer)

In [9]:
from app.vector_store import get_embeddings_batch

sample_texts = [
    "MLOps pipeline for automated model deployment and tracking.",
    "Generative AI and LangChain agentic workflow development."
]

try:
    vectors = get_embeddings_batch(sample_texts)
    print(f"✅ Success! Generated {len(vectors)} embedding vectors.")
    print(f"   Vector dimension size: {len(vectors[0])} dimensions.")
except Exception as e:
    print(f"❌ Embedding Generation Failed: {e}")

✅ Success! Generated 2 embedding vectors.
   Vector dimension size: 1536 dimensions.


## 3. Test Vector Store & Catalog Similarity Search

In [10]:
from app.database import get_all_products
from app.vector_store import VectorStoreManager

products = get_all_products()
print(f"📦 Loaded {len(products)} products from SQL Database.")

try:
    matches = VectorStoreManager.search_similar_products(
        query="MLOps CI/CD Python",
        top_k=3,
        products_catalog=products
    )
    print(f"✅ Success! Retrieved {len(matches)} matching courses for query:")
    for m in matches:
        print(f"   - [ID #{m['id']}] {m['title']} (${m['price']}) - {m['category']}")
except Exception as e:
    print(f"❌ Similarity Search Failed: {e}")

📦 Loaded 39 products from SQL Database.
✅ Success! Retrieved 3 matching courses for query:
   - [ID #11] MLOps for Real Teams ($179.0) - MLOps
   - [ID #45] Serverless AI Deployments on AWS Lambda & Modal ($169.0) - Cloud & DevOps
   - [ID #15] Evaluating LLM Applications ($149.0) - MLOps


## 4. Test Behavioral Event Logging (Telemetry Layer)

In [11]:
from app.database import record_events_batch, get_recent_user_events

test_user_id = 9999
sample_events = [
    {"event_type": "Click", "target_id": "1", "metadata": {"statement": "Interested in MLOps"}},
    {"event_type": "Search", "target_id": "search", "metadata": {"statement": "Python AI"}},
    {"event_type": "Dwell", "target_id": "2", "metadata": {"statement": "AI Architecture"}}
]

record_events_batch(test_user_id, sample_events)
recent = get_recent_user_events(test_user_id, limit=5)
print(f"✅ Recorded & retrieved {len(recent)} events for test user #{test_user_id}:")
for ev in recent:
    print(f"   - {ev['event_type']} on {ev['target_id']}: {ev.get('metadata')}")

✅ Recorded & retrieved 5 events for test user #9999:
   - Click on 1: None
   - Search on search: None
   - Dwell on 2: None
   - Click on 1: None
   - Search on search: None


## 5. Test AI Agent Full Recommendation & Re-Ranking

In [12]:
from app.agent import SmartRecoAgent

agent = SmartRecoAgent(user_id=test_user_id)
try:
    res = agent.generate_recommendation(force_refresh=True)
    print(f"✅ Success! AI Narrative:\n   '{res['narrative']}'\n")
    print(f"   Recommended Courses ({len(res['recommended_products'])}): ")
    for item in res['recommended_products']:
        print(f"   - {item['title']} (${item['price']})")
        print(f"     Reason: {item.get('ai_reason')}\n")
except Exception as e:
    print(f"❌ AI Agent Generation Failed: {e}")

✅ Success! AI Narrative:
   'Based on your interest in MLOps and Python AI, these two courses will enhance your skills significantly.'

   Recommended Courses (2): 
   - MLOps for Real Teams ($179.0)
     Reason: Focuses on CI/CD for models, aligning perfectly with your MLOps interest.

   - Evaluating LLM Applications ($149.0)
     Reason: Covers evaluation metrics for LLM applications, enhancing your understanding of AI performance.



## 6. Test Trace Log History

In [13]:
from app.agent import TRACE_LOGS, TRACES_LOCK

with TRACES_LOCK:
    traces = [t.model_dump() for t in TRACE_LOGS]

print(f"✅ Captured {len(traces)} recent execution traces:")
for t in traces:
    status_str = "✅ Success" if t["success"] else f"❌ Failed ({t.get('error_vector_db') or t.get('error_llm')})"
    print(f"   - Trace {t['trace_id'][:8]} | Total: {t['t_total_ms']:.1f}ms | Vector: {t['t_vector_ms']:.1f}ms | LLM: {t['t_llm_ms']:.1f}ms | {status_str}")

✅ Captured 1 recent execution traces:
   - Trace 82870558 | Total: 6593.2ms | Vector: 1212.7ms | LLM: 5211.6ms | ✅ Success
